In [1]:
import numpy as np
import pandas as pd
import torch

import sys
sys.path.append('../')
from utilities import binning_equal_q

In [2]:
def create_simulated_dataset(p_vector, p_constant=True, N=7, n_c=20, N_0=1e6, R=5e7, p=0.6, k=1e11, amp_rounds=26, seed=0):
    
    np.random.seed(seed)
    
    lengths = [torch.arange(n_c, dtype=torch.int8) for i in range(N)]
    all_seq = torch.cartesian_prod(*lengths)
    all_seq = all_seq.long()
    
    #first poisson sampling
    C = np.random.poisson(N_0*p_vector)
    
    #adjust dataset
    all_seq = all_seq[C > 0]
    C = C[C > 0]

    #binomial process
    if p_constant:
        for _ in range(amp_rounds):
            C += np.random.binomial(n=C, p=p)
    else:
        print('ciao')
        
    #second poisson sampling
    C = np.random.poisson(R*C/C.sum()) 
        
    #adjust dataset
    all_seq = all_seq[C > 0]
    C = C[C > 0]
    
    return all_seq.numpy(), C

In [3]:
def infer_fields(data, counts):
    
    fields = np.zeros([7,20])
    for pos in range(7):
        for color in range(20):
            fields[pos,color] = (counts*((data[:,pos] == color).astype('int'))).sum() 

    print(counts.sum(), fields.sum(1))

    fields = fields[:,:] / fields[:,0][:,np.newaxis]
    return torch.tensor(np.log(fields))

def log10p_vector_from_fields(fields):
    
    lengths = [torch.arange(20, dtype=torch.int8) for i in range(7)]
    all_seq = torch.cartesian_prod(*lengths)
    
    all_seq = all_seq.long()
    
    # generate p_vector
    p_vector = torch.zeros(20**7,dtype=torch.float32)

    for i in range(7):
        p_vector += fields[i,all_seq[:,i]]
        print(i)
    
    p_vector = torch.exp(p_vector)
    
    Z = torch.exp(fields).sum(1).prod()
    
    p_vector /= Z
    print(p_vector.sum())
    
    return torch.log(p_vector) / np.log(10)

def log10q_vector_func(data, fields):
    
    # generate q_vector
    q_vector = torch.zeros(len(data))

    for i in range(7):
        q_vector += fields[i,data[:,i]]
        print(i)

    q_vector = torch.exp(q_vector)

    Z = torch.exp(fields).sum(1).prod()

    q_vector /= Z

    print(q_vector.sum())
    return np.log10(q_vector.numpy())

In [4]:
data = pd.read_csv('../data/Byrne.csv',index_col=0).query('T0 > 0')
data = data.sort_values('T0', ascending=False)
data = data.iloc[1:,:]

counts = data['T0'].to_numpy()
data = data.iloc[:,:7].to_numpy()

In [5]:
fields = infer_fields(data, counts)

12195473 [12195473. 12195473. 12195473. 12195473. 12195473. 12195473. 12195473.]


In [6]:
data_log10p_vector = log10p_vector_from_fields(fields)

0
1
2
3
4
5
6
tensor(1.)


In [8]:
R = counts.sum()
F = np.loadtxt('../binning_real_data_q/F_Byrne.csv')[0]
N_0 = int(2*R / (F-1) / 1.6)
F, N_0

(8.922473368689328, 1924189)

In [9]:
for amp_rounds in range(0,30,5):

    print('amplification_rounds = %d'%amp_rounds)
    
    data, counts = create_simulated_dataset(p_vector= 10**(data_log10p_vector), R=R, N_0=N_0, amp_rounds=amp_rounds)
    fields = infer_fields(data, counts)
    log10p_vector = log10p_vector_from_fields(fields)
    sorted_log10p_vector = log10p_vector.sort()[0]
    log10q_vector = log10q_vector_func(data, fields)
    argsort = np.argsort(log10q_vector)
    sorted_log10q_vector = log10q_vector[argsort][::-1]
    counts = counts[argsort][::-1]
    
    bins=80
    df_bins = binning_equal_q(sorted_log10p_vector, sorted_log10q_vector, counts, bins=bins)#, writefolder='results_ByrneT0_IM', step=1)
    df_bins.to_csv('df_bins_simulation_Byrne_%damp_rounds.csv'%amp_rounds)
    
    del(sorted_log10p_vector)
    del(log10p_vector)

amplification_rounds = 0
12199410 [12199410. 12199410. 12199410. 12199410. 12199410. 12199410. 12199410.]
0
1
2
3
4
5
6
tensor(1.)
0
1
2
3
4
5
6
tensor(0.0081)
0
tensor(1279999992) tensor(1279618805)
elements in the bin: 381187
nonzeros: 23922
1
tensor(1279618805) tensor(1279044417)
elements in the bin: 574388
nonzeros: 23922
2
tensor(1279044417) tensor(1278346541)
elements in the bin: 697876
nonzeros: 23922
3
tensor(1278346541) tensor(1277533742)
elements in the bin: 812799
nonzeros: 23922
4
tensor(1277533742) tensor(1276624518)
elements in the bin: 909224
nonzeros: 23922
5
tensor(1276624518) tensor(1275607648)
elements in the bin: 1016870
nonzeros: 23922
6
tensor(1275607648) tensor(1274503342)
elements in the bin: 1104306
nonzeros: 23922
7
tensor(1274503342) tensor(1273296075)
elements in the bin: 1207267
nonzeros: 23922
8
tensor(1273296075) tensor(1272024818)
elements in the bin: 1271257
nonzeros: 23922
9
tensor(1272024818) tensor(1270636174)
elements in the bin: 1388644
nonzeros: 2

33
tensor(1209838807) tensor(1205628588)
elements in the bin: 4210219
nonzeros: 23251
34
tensor(1205628588) tensor(1201284780)
elements in the bin: 4343808
nonzeros: 23251
35
tensor(1201284780) tensor(1196762801)
elements in the bin: 4521979
nonzeros: 23251
36
tensor(1196762801) tensor(1192000543)
elements in the bin: 4762258
nonzeros: 23251
37
tensor(1192000543) tensor(1187074811)
elements in the bin: 4925732
nonzeros: 23251
38
tensor(1187074811) tensor(1181944916)
elements in the bin: 5129895
nonzeros: 23251
39
tensor(1181944916) tensor(1176676405)
elements in the bin: 5268511
nonzeros: 23251
40
tensor(1176676405) tensor(1171102545)
elements in the bin: 5573860
nonzeros: 23251
41
tensor(1171102545) tensor(1165367887)
elements in the bin: 5734658
nonzeros: 23251
42
tensor(1165367887) tensor(1159332476)
elements in the bin: 6035411
nonzeros: 23251
43
tensor(1159332476) tensor(1153133718)
elements in the bin: 6198758
nonzeros: 23251
44
tensor(1153133718) tensor(1146628223)
elements in t

47
tensor(1132707786) tensor(1125277790)
elements in the bin: 7429996
nonzeros: 23147
48
tensor(1125277790) tensor(1117515242)
elements in the bin: 7762548
nonzeros: 23147
49
tensor(1117515242) tensor(1109580968)
elements in the bin: 7934274
nonzeros: 23147
50
tensor(1109580968) tensor(1101223789)
elements in the bin: 8357179
nonzeros: 23147
51
tensor(1101223789) tensor(1092392818)
elements in the bin: 8830971
nonzeros: 23147
52
tensor(1092392818) tensor(1083254382)
elements in the bin: 9138436
nonzeros: 23147
53
tensor(1083254382) tensor(1073613034)
elements in the bin: 9641348
nonzeros: 23147
54
tensor(1073613034) tensor(1063526131)
elements in the bin: 10086903
nonzeros: 23147
55
tensor(1063526131) tensor(1052942267)
elements in the bin: 10583864
nonzeros: 23147
56
tensor(1052942267) tensor(1041745575)
elements in the bin: 11196692
nonzeros: 23147
57
tensor(1041745575) tensor(1030134860)
elements in the bin: 11610715
nonzeros: 23147
58
tensor(1030134860) tensor(1017924135)
elements 

61
tensor(991179099) tensor(976694366)
elements in the bin: 14484733
nonzeros: 23134
62
tensor(976694366) tensor(961332965)
elements in the bin: 15361401
nonzeros: 23134
63
tensor(961332965) tensor(945099588)
elements in the bin: 16233377
nonzeros: 23134
64
tensor(945099588) tensor(927501270)
elements in the bin: 17598318
nonzeros: 23134
65
tensor(927501270) tensor(909066657)
elements in the bin: 18434613
nonzeros: 23134
66
tensor(909066657) tensor(889045960)
elements in the bin: 20020697
nonzeros: 23134
67
tensor(889045960) tensor(867788531)
elements in the bin: 21257429
nonzeros: 23134
68
tensor(867788531) tensor(844545658)
elements in the bin: 23242873
nonzeros: 23134
69
tensor(844545658) tensor(818994648)
elements in the bin: 25551010
nonzeros: 23134
70
tensor(818994648) tensor(791108849)
elements in the bin: 27885799
nonzeros: 23134
71
tensor(791108849) tensor(760622038)
elements in the bin: 30486811
nonzeros: 23134
72
tensor(760622038) tensor(726759619)
elements in the bin: 33862

75
tensor(644568183) tensor(592140820)
elements in the bin: 52427363
nonzeros: 23139
76
tensor(592140820) tensor(529716477)
elements in the bin: 62424343
nonzeros: 23139
77
tensor(529716477) tensor(449557028)
elements in the bin: 80159449
nonzeros: 23139
78
tensor(449557028) tensor(333778035)
elements in the bin: 115778993
nonzeros: 23139
79
tensor(333778035) tensor(437264)
elements in the bin: 333340771
nonzeros: 23186
amplification_rounds = 25
12188323 [12188323. 12188323. 12188323. 12188323. 12188323. 12188323. 12188323.]
0
1
2
3
4
5
6
tensor(1.0000)
0
1
2
3
4
5
6
tensor(0.0078)
0
tensor(1279999991) tensor(1279619354)
elements in the bin: 380637
nonzeros: 23133
1
tensor(1279619354) tensor(1279044701)
elements in the bin: 574653
nonzeros: 23133
2
tensor(1279044701) tensor(1278347760)
elements in the bin: 696941
nonzeros: 23133
3
tensor(1278347760) tensor(1277534441)
elements in the bin: 813319
nonzeros: 23133
4
tensor(1277534441) tensor(1276627305)
elements in the bin: 907136
nonzero